In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(r"C:archive\hasyv2\hasy-data-labels.csv")
df_test = pd.read_csv(csv_path, nrows=1)
print(df_test.columns.tolist())

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
import torchvision
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# --- 設定 ---
base_path = Path(r"C:archive\hasyv2") 
csv_path = base_path / "hasy-data-labels.csv"
output_path = Path(r"C:math_dataset_cnn")

# 1. HASYv2 抽出・変換設定
target_latex = {
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4', '5': '5', '6': '6', '7': '7', '8': '8', '9': '9',
    '+': 'plus', '-': 'minus', '\\times': 'times', '\\div': 'divide',
    '<': 'less', '>': 'greater', '\\leq': 'leq', '\\geq': 'geq', 
    '\\cong': 'cong',
    'x': 'var_x', 'y': 'var_y', 'z': 'var_z', 'a': 'var_a', 'b': 'var_b', 'c': 'var_c',
    '\\alpha': 'alpha', '\\beta': 'beta',
    '\\infty': 'infty'
}
target_ids = {40: 'lp', 41: 'rp' }

def apply_augmentation(img_res, folder_name, idx):
    
    cv2.imwrite(str(output_path / folder_name / f"orig_{idx}.png"), img_res)

    if not folder_name.isdigit():
        kernel = np.ones((2, 2), np.uint8)
        
        bold_img = cv2.erode(img_res, kernel, iterations=1)
        cv2.imwrite(str(output_path / folder_name / f"bold_{idx}.png"), bold_img)

        shrink_ratio = 0.8
        h, w = 48, 48
        new_w, new_h = int(w * shrink_ratio), int(h * shrink_ratio)
        resized_small = cv2.resize(img_res, (new_w, new_h), interpolation=cv2.INTER_AREA)
        small_img = np.full((h, w), 255, dtype=np.uint8)
        small_img[(h-new_h)//2 : (h-new_h)//2+new_h, (w-new_w)//2 : (w-new_w)//2+new_w] = resized_small
        cv2.imwrite(str(output_path / folder_name / f"small_{idx}.png"), small_img)

        resized_bold_small = cv2.resize(bold_img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        bold_small_img = np.full((h, w), 255, dtype=np.uint8)
        bold_small_img[(h-new_h)//2 : (h-new_h)//2+new_h, (w-new_w)//2 : (w-new_w)//2+new_w] = resized_bold_small
        cv2.imwrite(str(output_path / folder_name / f"bold_small_{idx}.png"), bold_small_img)

def collect_hasy():
    print("--- HASYv2の抽出と拡張を開始 ---")
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    
    all_folders = set(list(target_latex.values()) + list(target_ids.values()))
    for f in all_folders: (output_path / f).mkdir(parents=True, exist_ok=True)

    for lx, folder in target_latex.items():
        matches = df[df['latex'] == lx]
        for idx, row in matches.iterrows():
            src = base_path / row['path'].strip()
            img = cv2.imread(str(src), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img_res = cv2.resize(img, (48, 48), interpolation=cv2.INTER_AREA)
                apply_augmentation(img_res, folder, idx)


    for sid, folder in target_ids.items():
        matches = df[df['symbol_id'] == sid]
        for idx, row in matches.iterrows():
            src = base_path / row['path'].strip()
            img = cv2.imread(str(src), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img_res = cv2.resize(img, (48, 48), interpolation=cv2.INTER_AREA)
                apply_augmentation(img_res, folder, idx)

def collect_mnist():
    print("\n--- MNISTの反転・統合を開始 ---")
    mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True)
    counts = [0] * 10
    limit = 6000 

    for img, label in tqdm(mnist_train, total=len(mnist_train)):
        if counts[label] < limit:
            img_np = np.array(img)
            inverted_img = 255 - img_np # 白背景・黒文字へ
            final_img = Image.fromarray(inverted_img).resize((48, 48), Image.BICUBIC)
            save_path = output_path / str(label) / f"mnist_inv_{counts[label]}.png"
            final_img.save(save_path)
            counts[label] += 1
        if sum(counts) >= limit * 10: break

if __name__ == "__main__":
    collect_hasy()
    collect_mnist()
    print(f"\nすべての処理が完了しました！ 保存先: {output_path}")

In [ ]:
# 上下左右反転できる追加の画像を拡張
# =や+など
#48x48のpng形式の画像を生成

import cv2
import numpy as np
import random
from pathlib import Path

# 追加の画像フォルダーのパスを調整してください
input_dir = Path(r"") #例(拡張前):plus
output_dir = Path(r"")#例(拡張後):math_dataset_cnn\plus
output_dir.mkdir(parents=True, exist_ok=True)

TARGET_COUNT = 6000
IMG_SIZE = 48

def augment_image(img):
    #1枚の画像に対して指定の加工
    h, w = img.shape[:2]
    
    # 1. 左右反転 (50%の確率)
    if random.random() > 0.5:
        img = cv2.flip(img, 1)

    # 2. 上下反転 (50%の確率)
    if random.random() > 0.5:
        img = cv2.flip(img, 0)

    # 3. 角度の変化 (+-10度)
    angle = random.uniform(-10, 10)
    matrix = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img = cv2.warpAffine(img, matrix, (w, h), borderValue=255)

    # 4. フォントサイズの減少 (縦幅・横幅それぞれ 0~20% 減少)
    # scale=0.8 が 20%減少に相当
    scale_w = random.uniform(0.3, 1.0)
    scale_h = random.uniform(0.8, 1.0)
    
    new_w, new_h = int(w * scale_w), int(h * scale_h)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 中央に再配置
    canvas = np.full((h, w), 255, dtype=np.uint8)
    pad_x = (w - new_w) // 2
    pad_y = (h - new_h) // 2
    canvas[pad_y:pad_y+new_h, pad_x:pad_x+new_w] = resized
    
    return canvas

def main():
    # 元画像のリスト取得
    original_files = list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))
    if not original_files:
        print(f"元画像が見つかりません: {input_dir}")
        return

    print(f"元画像 {len(original_files)} 枚から {TARGET_COUNT} 枚を生成します...")

    count = 0
    while count < TARGET_COUNT:
        for file_path in original_files:
            if count >= TARGET_COUNT:
                break
            
            img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            
            # リサイズ
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            # 加工実行
            aug_img = augment_image(img)

            # 保存
            save_name = f"x_aug_{count:04d}.png"
            cv2.imwrite(str(output_dir / save_name), aug_img)
            
            count += 1
            if count % 500 == 0:
                print(f"進捗: {count} / {TARGET_COUNT}")

    print(f"完了！ 保存先: {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
# 上下左右反転できない追加の画像を拡張
# πや7など
#48x48のpng形式の画像を生成

import cv2
import numpy as np
import random
from pathlib import Path

# --- 設定 ---
input_dir = Path(r"") #例(拡張前):pi
output_dir = Path(r"")#例(拡張後):math_dataset_cnn\pi
output_dir.mkdir(parents=True, exist_ok=True)

TARGET_COUNT = 6000
IMG_SIZE = 48

def augment_image(img):
    #1枚の画像に対してランダムな加工を施す
    h, w = img.shape[:2]
    
    # 1. 角度の変化 (+-10度)
    angle = random.uniform(-10, 10)
    matrix = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img = cv2.warpAffine(img, matrix, (w, h), borderValue=255)

    # 2. サイズの変更 (0.8倍〜1.2倍)
    scale_w = random.uniform(0.7, 1.2)
    scale_h = random.uniform(0.8, 1.2)
    
    new_w, new_h = int(w * scale_w), int(h * scale_h)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # --- 新しいキャンバスを一旦作り、リサイズ後の画像を中央に合成する ---
    # 白背景（255）の 48x48 キャンバスを作成
    canvas = np.full((h, w), 255, dtype=np.uint8)
    y_offset = max(0, (h - new_h) // 2)
    x_offset = max(0, (w - new_w) // 2)
    
    src_y_start = max(0, (new_h - h) // 2) if new_h > h else 0
    src_x_start = max(0, (new_w - w) // 2) if new_w > w else 0
    
    copy_h = min(h, new_h)
    copy_w = min(w, new_w)
    
    # 合成実行
    canvas[y_offset:y_offset+copy_h, x_offset:x_offset+copy_w] = \
        resized[src_y_start:src_y_start+copy_h, src_x_start:src_x_start+copy_w]
    
    img = canvas

    # 3. 太さの変更 (膨張・収縮)
    img_inv = cv2.bitwise_not(img)
    thickness_choice = random.choice(['thicker', 'thinner', 'normal'])
    kernel = np.ones((2, 2), np.uint8)
    
    if thickness_choice == 'thicker':
        img_inv = cv2.dilate(img_inv, kernel, iterations=1)
    elif thickness_choice == 'thinner':
        img_inv = cv2.erode(img_inv, kernel, iterations=1)
    
    img = cv2.bitwise_not(img_inv)

    return img

def main():
    # 元画像のリスト取得
    original_files = list(input_dir.glob("*.jpg")) + list(input_dir.glob("*.png"))
    if not original_files:
        print("元画像が見つかりません。パスを確認してください。")
        return

    print(f"元画像 {len(original_files)} 枚から {TARGET_COUNT} 枚を生成します...")

    count = 0
    while count < TARGET_COUNT:
        for file_path in original_files:
            if count >= TARGET_COUNT:
                break
            
            # 画像読み込み
            img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            
            # リサイズ
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            # 加工
            aug_img = augment_image(img)

            # 保存
            save_name = f"pi_aug_{count:04d}.png"
            cv2.imwrite(str(output_dir / save_name), aug_img)
            
            count += 1
            if count % 500 == 0:
                print(f"進捗: {count} / {TARGET_COUNT}")

    print(f"完了！ 保存先: {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
import torch
from torch import nn

class UniversalMathCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) 
        self.pool = nn.MaxPool2d(2, 2)
        
        # 48x48 -> (pool1) 24x24 -> (pool2) 12x12 -> (pool3) 6x6
        # 128チャネル * 6 * 6 = 4608
        self.fc1 = nn.Linear(128 * 6 * 6, 256) 
        self.fc2 = nn.Linear(256, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25) # 過学習防止

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1) 
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# 1. 48x48にリサイズし、正規化
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 2. データセットの読み込み
# フォルダ構成: math_dataset_cnn/クラス名/画像.png
full_dataset = ImageFolder(root=r'\math_dataset_cnn', transform=transform)

# クラス名の確認
class_names = full_dataset.classes
num_classes = len(class_names)
print(f"検出されたクラス数: {num_classes}")
print(f"クラス一覧: {class_names}")

# 訓練データと検証データに分割 (9:1)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# 3. モデルの定義 (数式用CNN)
class MathFormulaCNN(nn.Module):
    def __init__(self, num_classes):
        super(MathFormulaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        # 48x48 -> (pool1) 24x24 -> (pool2) 12x12 -> (pool3) 6x6
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# デバイス設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MathFormulaCNN(num_classes).to(device)

# 4. 損失関数と最適化
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 学習ループ
epochs = 15 #ループ回数
history = {'train_loss': [], 'val_acc': []}

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # 検証
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    history['train_loss'].append(running_loss / len(train_loader))
    history['val_acc'].append(accuracy)
    
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, Val Acc: {accuracy:.2f}%')

# モデルの保存
torch.save(model.state_dict(), 'math_sunpul_model.pth')
# クラス名リストも保存
import json
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)

print("学習完了！ 'math_sunpul_model.pth' として保存されました。")

# --- グラフの表示 ---

# 横に2つ並べる
plt.figure(figsize=(12, 5))

# 1. 損失の推移(Loss)
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), history['train_loss'], marker='o', label='Train Loss')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

# 2. 精度の推移(Accuracy)
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), history['val_acc'], marker='o', color='orange', label='Val Acc')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.tight_layout() # レイアウトを整える
plt.show()

# 最終結果のサマリーを表示
print(f"最終 Epoch の Loss: {history['train_loss'][-1]:.4f}")
print(f"最終 Epoch の Accuracy: {history['val_acc'][-1]:.2f}%")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json

class MathFormulaCNN(nn.Module):
    def __init__(self, num_classes):
        super(MathFormulaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1) 
        self.pool = nn.MaxPool2d(2, 2)
        
        # 48x48 -> pool1:24x24 -> pool2:12x12 -> pool3:6x6
        # 128ch * 6 * 6 = 4608
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, num_classes) 
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

JSON_PATH = 'class_names.json'
with open(JSON_PATH, 'r') as f:
    class_names = json.load(f)
num_classes = len(class_names)

model = MathFormulaCNN(num_classes)
SAVE_PATH = r"math_universal_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 重みの読み込み
model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
model.to(device)
model.eval()

print(f"読み込みが完了しました。クラス数: {num_classes}")
print(f"対応記号: {class_names}")

読み込みが完了しました。クラス数: 30
対応記号: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'alpha', 'beta', 'cong', 'divide', 'equal', 'geq', 'greater', 'infty', 'leq', 'less', 'minus', 'pi', 'plus', 'times', 'var_a', 'var_b', 'var_c', 'var_x', 'var_y', 'var_z']


In [ ]:
#テスト
#手書きの計算式(一行)を読み込み
import cv2
import numpy as np
from pathlib import Path
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch
import json

# --- 設定 ---
folder_path = Path(r"")#テスト用フォルダーパス
file_name = "" #テスト用画像名
target_file = folder_path / file_name

# クラス名の読み込み
with open('class_names.json', 'r') as f:
    class_names = json.load(f)

def resize_with_padding_white(img, size=(48, 48)):
    #アスペクト比を維持したまま、余白を追加して正方形にする
    h, w = img.shape
    # 1. ターゲットサイズより少し小さめにリサイズ
    # 内側に少しマージンを持たせる(px)
    margin = 8
    target_w, target_h = size[0] - margin, size[1] - margin
    
    # アスペクト比を維持するスケール計算
    scale = min(target_w / w, target_h / h)
    
    new_w, new_h = int(w * scale), int(h * scale)
    # リサイズ実行
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # 2. 48x48のキャンバスを作成
    canvas = np.full(size, 255, dtype=np.uint8)
    
    # 3. キャンバスの中央に配置
    top = (size[1] - new_h) // 2
    left = (size[0] - new_w) // 2
    canvas[top:top+new_h, left:left+new_w] = resized
    
    return canvas


def get_prediction(image_np, model, device):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    img_t = transform(image_np).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        out = model(img_t)
        probs = torch.nn.functional.softmax(out, dim=1)
        conf, pred = torch.max(probs, 1)
    return pred.item(), conf.item()

def merge_boxes(bboxes, threshold_dist=15): 
    if not bboxes: return []
    bboxes = sorted(bboxes, key=lambda x: x[0])
    merged = []
    while len(bboxes) > 0:
        curr = list(bboxes.pop(0))
        found_merge = True
        while found_merge:
            found_merge = False
            for i in range(len(bboxes)):
                next_box = bboxes[i]
                overlap_x = max(0, min(curr[0]+curr[2], next_box[0]+next_box[2]) - max(curr[0], next_box[0]))
                dist_x = max(0, next_box[0] - (curr[0] + curr[2]), curr[0] - (next_box[0] + next_box[2]))
                dist_y = max(0, next_box[1] - (curr[1] + curr[3]), curr[1] - (next_box[1] + next_box[3]))
                is_vertical_stack = overlap_x > min(curr[2], next_box[2]) * 0.5
                if is_vertical_stack:
                    v_threshold = 25
                    h_threshold = 5
                else:
                    v_threshold = threshold_dist
                    h_threshold = 5 
                if dist_x < h_threshold and dist_y < v_threshold:
                    new_x = min(curr[0], next_box[0])
                    new_y = min(curr[1], next_box[1])
                    new_w = max(curr[0]+curr[2], next_box[0]+next_box[2]) - new_x
                    new_h = max(curr[1]+curr[3], next_box[1]+next_box[3]) - new_y
                    curr = [new_x, new_y, new_w, new_h]
                    bboxes.pop(i)
                    found_merge = True
                    break
        merged.append(tuple(curr))
    return merged

if target_file.exists():
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        img_cv = cv2.imread(str(target_file), cv2.IMREAD_GRAYSCALE)
        _, thresh_temp = cv2.threshold(img_cv, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        coords = np.column_stack(np.where(thresh_temp > 0))
        
        if len(coords) > 0:
            angle = cv2.minAreaRect(coords)[-1]
            if angle < -45: angle = -(90 + angle)
            elif angle > 45: angle = 90 - angle
            else: angle = -angle
            (h, w) = img_cv.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            img_cv = cv2.warpAffine(img_cv, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=255)

        img_cv = cv2.GaussianBlur(img_cv, (3, 3), 0)
        img_cv = cv2.medianBlur(img_cv, 3)
        thresh = cv2.adaptiveThreshold(img_cv, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 45, 30)
        
        thresh_for_cnt = cv2.bitwise_not(thresh)
        kernel_connect = np.ones((3, 3), np.uint8) 
        thresh_for_cnt = cv2.dilate(thresh_for_cnt, kernel_connect, iterations=1)
        thresh_for_cnt = cv2.erode(thresh_for_cnt, kernel_connect, iterations=1)
        
        contours, _ = cv2.findContours(thresh_for_cnt, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        initial_bboxes = [cv2.boundingRect(cnt) for cnt in contours if cv2.contourArea(cnt) > 10]
        
        temp_bboxes = merge_boxes(initial_bboxes, threshold_dist=15)
        temp_bboxes = sorted(temp_bboxes, key=lambda x: x[0])

        if len(temp_bboxes) == 0:
            print("文字が検出されませんでした。")
        else:
            avg_width = np.mean([b[2] for b in temp_bboxes])
            
            final_bboxes = []
            for (x, y, w, h) in temp_bboxes:
                if w > avg_width * 2.0 and w > h * 1.5:
                    roi_to_split = thresh[y:y+h, x:x+w]
                    inv_roi = cv2.bitwise_not(roi_to_split)
                    projection = np.sum(inv_roi, axis=0)
                    search_s, search_e = int(w * 0.3), int(w * 0.7)
                    split_rel = search_s + np.argmin(projection[search_s:search_e])
                    parts = [(x, y, split_rel, h), (x + split_rel, y, w - split_rel, h)]
                    for px, py, pw, ph in parts:
                        part_roi = thresh[py:py+ph, px:px+pw]
                        black_pixels = np.sum(part_roi == 0)
                        if pw > avg_width * 0.4 and black_pixels > 30:
                            final_bboxes.append((px, py, pw, ph))
                else:
                    final_bboxes.append((x, y, w, h))

            # --- 1. 一次推論 & べき乗判定 ---
            final_symbols = []
            rois_to_show = []
            is_superscripts = []
            prev_bbox = None 
            prev_symbol = None  

            for i, (x, y, w, h) in enumerate(final_bboxes):
                roi = thresh[y:y+h, x:x+w]
                roi_display = resize_with_padding_white(roi)
                rois_to_show.append(roi_display)

                idx, confidence = get_prediction(roi_display, model, device)
                symbol = str(class_names[idx])

                # べき乗判定
                is_super = False
                if prev_bbox is not None:
                    px, py, pw, ph = prev_bbox
                    current_center_y = y + h / 2
                    prev_center_y = py + ph / 2
                    if (y + h) < (py + ph * 0.6) or (current_center_y < prev_center_y - ph * 0.3):
                        if h < ph * 1.1: is_super = True
                
                # 連続文字などの文脈補正
                if symbol == 'var_z' and prev_symbol == 'var_z':
                    if final_symbols: final_symbols[-1] = '2'
                elif symbol == 'var_y' and prev_symbol == 'var_y':
                    if final_symbols: final_symbols[-1] = '7'
                
                # 数字の前の変数補正
                elif not is_super and symbol.isdigit() and prev_symbol in ['alpha', 'beta','var_a', 'var_b', 'var_c','var_x', 'var_y', 'var_z']:
                    prev_roi = rois_to_show[-2]
                    img_t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])(prev_roi).unsqueeze(0).to(device)
                    model.eval()
                    with torch.no_grad():
                        out = model(img_t)
                        digit_logits = out[0, 0:10] 
                        new_digit_idx = torch.argmax(digit_logits).item()
                    if final_symbols: final_symbols[-1] = class_names[new_digit_idx]

                is_superscripts.append(is_super)
                final_symbols.append(symbol)
                prev_bbox = (x, y, w, h)
                prev_symbol = final_symbols[-1]

            # --- 2. 式に '=' が含まれる場合、その前の不等号を数字に修正 ---
            # 不等号として認識されやすいクラス名のリスト
            inequality_classes = ['geq', 'greater','leq', 'less']
            
            if 'equal' in final_symbols or '=' in final_symbols:
                try:
                    eq_idx = next(i for i, s in enumerate(final_symbols) if s in ['equal', '='])
                    for i in range(eq_idx):
                        if final_symbols[i] in inequality_classes:
                            target_roi = rois_to_show[i]
                            img_t = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])(target_roi).unsqueeze(0).to(device)
                            model.eval()
                            with torch.no_grad():
                                out = model(img_t)
                                digit_logits = out[0, 0:10] 
                                new_digit_idx = torch.argmax(digit_logits).item()
                            final_symbols[i] = class_names[new_digit_idx]
                except StopIteration:
                    pass

            # --- 特定記号の並びを補正 (*+ -> x+) ---
            for i in range(len(final_symbols) - 1):
                # 現在が 'times' で、次が 'plus' の場合
                current_sym = final_symbols[i]
                next_sym = final_symbols[i+1]
                
                if current_sym in ['*', 'times'] and next_sym in ['+', 'plus']:
                    # 強制的に 'var_x' に書き換え
                    final_symbols[i] = 'var_x'

            # 1枚あたりのサイズを少し大きく設定
            plt.figure(figsize=(max(len(final_bboxes) * 2.0, 10), 5))
            
            for i in range(len(final_bboxes)):
                display_char = final_symbols[i].replace('var_x','x').replace('var_y','y').replace('var_z','z')
                sup_flag = "^" if is_superscripts[i] else ""
                
                # サブプロット作成
                ax = plt.subplot(1, len(final_bboxes), i + 1)
                
                # 1. 実際のモデル入力画像（rois_to_show[i]）を表示
                # 枠線を見やすくするため、cmap='gray_r' (白黒反転) で表示します。
                # (文字が白、背景が黒になり、黒枠が映えます)
                plt.imshow(rois_to_show[i], cmap='gray')
                
                # タイトル設定
                plt.title(f'{display_char}{sup_flag}', fontsize=16, fontweight='bold')
                
                
                for spine in ax.spines.values():
                    spine.set_visible(True)
                    spine.set_color('black') # 枠線の色
                    spine.set_linewidth(3)   # 枠線の太さ
                
                plt.xticks([])
                plt.yticks([])
                
            # レイアウトを自動調整して表示
            plt.tight_layout()
            plt.show()
            
            # 最終結果
            final_formula = ""
            for i in range(len(final_symbols)):
                char = final_symbols[i].replace('var_x','x').replace('var_y','y').replace('var_z','z')
                if is_superscripts[i]: final_formula += "^"
                final_formula += char
            final_formula = final_formula.replace('plus', '+').replace('equal', '=').replace('minus', '-')
            print(f"認識結果: {final_formula}")

    except Exception as e:
        print(f"エラーが発生しました: {e}")

In [ ]:
#自分の手書き式に対応させるため間違いを追加学習する

import cv2
import numpy as np
from pathlib import Path
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import json

# --- 記号の入力補完マップ ---
# 人間が入力する文字から class_names の名前に変換します
INPUT_MAP = {
    "=": "equal",
    "+": "plus",
    "-": "minus",
    "*": "times",
    "x": "var_x",
    "y": "var_y",
    "z": "var_z",
    "a": "var_a",
    "b": "var_b",
    "c": "var_c"
}

# --- 1. モデル定義 ---
class MathFormulaCNN(nn.Module):
    def __init__(self, num_classes):
        super(MathFormulaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def resize_with_padding_white(img, size=(48, 48)):
    h, w = img.shape
    margin = 8
    target_w, target_h = size[0] - margin, size[1] - margin
    scale = min(target_w / w, target_h / h)
    new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.full(size, 255, dtype=np.uint8)
    top, left = (size[1] - new_h) // 2, (size[0] - new_w) // 2
    canvas[top:top+new_h, left:left+new_w] = resized
    return canvas

def merge_boxes(bboxes, threshold_dist=15):
    if not bboxes: return []
    bboxes = sorted(bboxes, key=lambda x: x[0])
    merged = []
    while len(bboxes) > 0:
        curr = list(bboxes.pop(0))
        found_merge = True
        while found_merge:
            found_merge = False
            for i in range(len(bboxes)):
                next_box = bboxes[i]
                overlap_x = max(0, min(curr[0]+curr[2], next_box[0]+next_box[2]) - max(curr[0], next_box[0]))
                dist_x = max(0, next_box[0] - (curr[0] + curr[2]), curr[0] - (next_box[0] + next_box[2]))
                dist_y = max(0, next_box[1] - (curr[1] + curr[3]), curr[1] - (next_box[1] + next_box[3]))
                is_vertical_stack = overlap_x > min(curr[2], next_box[2]) * 0.5
                v_threshold, h_threshold = (25, 5) if is_vertical_stack else (threshold_dist, 5)
                if dist_x < h_threshold and dist_y < v_threshold:
                    new_x = min(curr[0], next_box[0])
                    new_y = min(curr[1], next_box[1])
                    new_w = max(curr[0]+curr[2], next_box[0]+next_box[2]) - new_x
                    new_h = max(curr[1]+curr[3], next_box[1]+next_box[3]) - new_y
                    curr = [new_x, new_y, new_w, new_h]
                    bboxes.pop(i)
                    found_merge = True
                    break
        merged.append(tuple(curr))
    return merged

def update_model(image_np, correct_label_idx, model, device):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    img_t = transform(image_np).unsqueeze(0).to(device)
    label_t = torch.tensor([correct_label_idx]).to(device)
    for _ in range(5):
        optimizer.zero_grad(); loss = criterion(model(img_t), label_t)
        loss.backward(); optimizer.step()

# --- メイン処理 ---
folder_path = Path(r"")#テスト用フォルダーパス
file_name = "" #テスト用画像名
target_file = folder_path / file_name
model_path = 'Math_universal_model.pth'

with open('class_names.json', 'r') as f:
    class_names = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MathFormulaCNN(len(class_names)).to(device)
if Path(model_path).exists():
    model.load_state_dict(torch.load(model_path, map_location=device))

if target_file.exists():
    img_cv = cv2.imread(str(target_file), cv2.IMREAD_GRAYSCALE)
    # 回転補正
    _, t_temp = cv2.threshold(img_cv, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(t_temp > 0))
    if len(coords) > 0:
        angle = cv2.minAreaRect(coords)[-1]
        angle = -(90 + angle) if angle < -45 else (90 - angle if angle > 45 else -angle)
        M = cv2.getRotationMatrix2D((img_cv.shape[1]//2, img_cv.shape[0]//2), angle, 1.0)
        img_cv = cv2.warpAffine(img_cv, M, (img_cv.shape[1], img_cv.shape[0]), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=255)

    img_cv = cv2.GaussianBlur(img_cv, (3, 3), 0)
    thresh = cv2.adaptiveThreshold(img_cv, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 45, 30)
    
    # 結合 & 再分割
    cnts, _ = cv2.findContours(cv2.bitwise_not(thresh), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ini_boxes = [cv2.boundingRect(c) for c in cnts if cv2.contourArea(c) > 10]
    m_boxes = merge_boxes(ini_boxes)
    
    final_bboxes = []
    if m_boxes:
        avg_w = np.mean([b[2] for b in m_boxes])
        for (x, y, w, h) in sorted(m_boxes, key=lambda b: b[0]):
            if w > avg_w * 2.0 and w > h * 1.5:
                roi = thresh[y:y+h, x:x+w]
                proj = np.sum(cv2.bitwise_not(roi), axis=0)
                split = int(w*0.3) + np.argmin(proj[int(w*0.3):int(w*0.7)])
                final_bboxes.extend([(x,y,split,h), (x+split,y,w-split,h)])
            else: final_bboxes.append((x,y,w,h))

    # 推論 & 補正
    final_symbols, rois_for_learning, is_supers = [], [], []
    prev_bbox, prev_sym = None, None
    model.eval()

    for x, y, w, h in final_bboxes:
        roi_norm = resize_with_padding_white(thresh[y:y+h, x:x+w])
        rois_for_learning.append(roi_norm)
        with torch.no_grad():
            out = model(transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,),(0.5,))])(roi_norm).unsqueeze(0).to(device))
            idx = torch.argmax(out).item()
        
        sym = class_names[idx]
        is_sup = False
        if prev_bbox:
            if (y+h) < (prev_bbox[1]+prev_bbox[3]*0.6) or (y+h/2 < prev_bbox[1]+prev_bbox[3]/2 - prev_bbox[3]*0.3):
                if h < prev_bbox[3]*1.1: is_sup = True
        
        # 文脈補正
        if sym == 'var_z' and prev_sym == 'var_z': sym = '2'
        elif sym == 'var_y' and prev_sym == 'var_y': sym = '7'
        elif not is_sup and sym.isdigit() and prev_sym in ['var_x','var_y','var_z']:
            sym = class_names[torch.argmax(out[0, 0:10]).item()]
        
        final_symbols.append(sym); is_supers.append(is_sup)
        prev_bbox, prev_sym = (x,y,w,h), sym

    # 最終出力表示
    formula = "".join([("^"+s if sp else s) for s, sp in zip(final_symbols, is_supers)])
    formula = formula.replace('var_x','x').replace('var_y','y').replace('var_z','z').replace('plus','+').replace('equal','=').replace('minus','-')
    print(f"\n最終結果: {formula}\n")

    # --- 学習フェーズの修正箇所 ---
    print("\n" + "="*30)
    print("【学習・修正フェーズ】")
    print("・Enterキーのみ  ： 正解（そのまま次へ）")
    print("・文字/記号を入力： 修正して学習（y, x, =, +, - など）")
    print("・スペースキー   ： スキップ（小数点など）")
    print("="*30)

    for i in range(len(final_symbols)):
        # 画像表示
        plt.imshow(rois_for_learning[i], cmap='gray')
        plt.title(f"Target Symbol [{i+1}/{len(final_symbols)}]: {final_symbols[i]}")
        plt.show(block=False)
        plt.pause(0.3)

        # 入力待機
        raw_in = input(f"[{i+1}/{len(final_symbols)}] AI結果: '{final_symbols[i]}' -> 修正(or Enter): ")
        
        # 1. Enterキーのみ（空文字）の場合
        if raw_in == "":
            # 何もせず次の文字へ
            print(">> OKとして確定")
            pass 
            
        # 2. スペースキーの場合（覚えさせていない小数点のドットなど）
        elif raw_in == " ":
            print(">> 学習をスキップしました")
            
        # 3. 何か文字が入力された場合
        else:
            # 入力された記号（=, +, x, yなど）を内部名に変換
            target_name = INPUT_MAP.get(raw_in, raw_in)
            
            if target_name in class_names:
                correct_idx = class_names.index(target_name)
                update_model(rois_for_learning[i], correct_idx, model, device)
                print(f">> '{target_name}' として修正学習を完了しました。")
            else:
                print(f"!! エラー: '{target_name}' は class_names.json に見当たりません。")
        
        plt.close()

    # 最後にモデルを保存
    torch.save(model.state_dict(), model_path)
    print("\nすべての確認が終了し、モデルを保存しました。")